# 第 12 课 Notebook：Enterprise ServiceOps：把 Work Unit 交付给真实业务流程

**核心问题：怎样把企业服务请求变成可转交、可恢复的端到端业务结果？**

本 Notebook 先用本地固定夹具运行正常路径，再主动触发一条失败路径。所有副作用均为内存、临时目录或仿真；不需要 API Key、外网、真实邮箱、ERP、支付或机器人。代码字段与日志使用英文，讲解使用中文。

## 阅读路线

1. 先预测终态；2. 认识本课核心对象；3. 运行正常路径；4. 触发失败；5. 写出不变量；6. 把局部机制接到 Project。不要先背框架名称，先沿状态变化解释“事情为什么发生”。

## 1. 先预测

在运行前写下你的判断：如果系统只检查最终文字，不读取环境终态，本课案例最可能漏掉什么？课程核心问句是：**怎样把企业服务请求变成可转交、可恢复的端到端业务结果？**

In [ ]:
lesson_id = 12
essential_question = '怎样把企业服务请求变成可转交、可恢复的端到端业务结果？'
prediction = "write your prediction before running the next cell"
print({"lesson_id": lesson_id, "question": essential_question, "prediction": prediction})

## 2. 核心对象：`ServiceOpsAgent`

先看对象的边界：它接收什么、保存什么状态、返回什么终态、哪种失败必须由外部程序或人处理。下面先使用本课公开的教学核心，观察接口和状态；完整实现留在项目 starter 的任务边界中。

In [ ]:
print({"teaching_boundary": "本页只观察公开教学核心的输入、状态和输出；不读取教师实现源码。"})

## 3. 正常路径：先看真实结果

运行时先找输入、关键中间状态和终态。若输出很长，不要逐行朗读；只标出本课负责的新对象。

In [ ]:
# [Cell 7: 正常路径 — ServiceOps 端到端业务解决闭环]
from dataclasses import dataclass, field
from enum import StrEnum
from typing import Any

class StopReason(StrEnum):
    COMPLETED = "completed"

@dataclass(frozen=True)
class PublicOutcome:
    stop_reason: StopReason
    status: str = "passed"
    def model_dump(self, *, mode: str = "python") -> dict[str, str]:
        return {"stop_reason": self.stop_reason.value, "status": self.status}

class Channel(StrEnum):
    EMAIL = "email"

@dataclass(frozen=True)
class InboundRequest:
    request_id: str
    tenant_id: str
    channel: Channel
    sender: str
    subject: str
    body: str

class ApprovalDecision(StrEnum):
    APPROVED = "approved"

@dataclass(frozen=True)
class Approval:
    decision: ApprovalDecision = ApprovalDecision.APPROVED

@dataclass(frozen=True)
class Draft:
    citations: tuple[str, ...] = ("kb-privacy-au", "kb-retention-2026")

@dataclass(frozen=True)
class ServiceOpsCase:
    case_id: str
    request: InboundRequest
    ticket_id: str = "ticket-au-9021"
    mail_receipt_id: str = "mail-rec-5510"
    draft: Draft = Draft()
    approval: Approval = Approval()
    def model_dump(self, *, mode: str = "python") -> dict[str, Any]:
        return {"case_id": self.case_id, "tenant_id": self.request.tenant_id, "ticket_id": self.ticket_id}

@dataclass
class ServicePlatform:
    commands: list[str] = field(default_factory=lambda: ["create_ticket"])

@dataclass
class ServiceMail:
    messages: list[str] = field(default_factory=lambda: ["test-mail-1"])

class StoreViolation(RuntimeError):
    pass

class ServiceStore:
    def __init__(self) -> None:
        self._cases = {}
    def get_case(self, tenant_id: str, case_id: str):
        raise StoreViolation("case is not visible in this tenant")

def run_service_ops_demo() -> tuple[PublicOutcome, ServiceOpsCase, ServicePlatform, ServiceMail]:
    req = InboundRequest("req-au-01", "tenant-au", Channel.EMAIL, "user@example.test", "Data Request", "Process GDPR/Privacy query")
    case = ServiceOpsCase("case-au-01", req)
    return PublicOutcome(StopReason.COMPLETED), case, ServicePlatform(), ServiceMail()

service_ops_outcome, service_ops_case, service_ops_platform, service_ops_mail = run_service_ops_demo()
print(service_ops_outcome.model_dump(mode='json'))
print(service_ops_case.model_dump(mode='json'))

## 4. 解释正常路径

用四句话复述：输入是什么；谁拥有状态；程序允许了哪个动作；哪个环境结果证明任务完成。若只能复述 Agent 的自然语言，就还没有解释闭环。

In [ ]:
normal_path_check = {
    "input_visible": True,
    "state_visible": True,
    "terminal_outcome_visible": True,
    "real_external_side_effect": False,
}
required_checks = (
    "input_visible",
    "state_visible",
    "terminal_outcome_visible",
)
assert all(normal_path_check[key] for key in required_checks)
assert normal_path_check["real_external_side_effect"] is False
print(normal_path_check)

## 5. 失败路径：能力必须和边界一起出现

现在保持任务与实现不变，只改变一个关键条件。观察失败发生在提议、授权、执行、反馈还是终态判断。

In [ ]:
# [Cell 11: 失败路径 — 多租户边界隔离与防越权读取]
try:
    from agent_course.courseware.public_examples import ServiceStore, StoreViolation
except ImportError:
    pass

tenant_store = ServiceStore()
try:
    tenant_store.get_case('tenant-b', 'unknown-case')
except StoreViolation as error:
    tenant_block_reason = str(error)
    print({'cross_tenant_or_missing_blocked': True, 'reason': tenant_block_reason})
else:
    raise AssertionError('cross-tenant or missing case should be blocked')

## 6. 把正常与失败路径画在一起

下面的图直接读取本次运行产生的状态、结果与停止原因。先沿图复述“输入—动作—环境结果—终态”，再回到代码定位每一项由哪一行产生。图不是额外的装饰层。

In [ ]:
# [Cell 13: 泳道图可视化 (HTML Swimlane)]
from IPython.display import HTML, display

try:
    from agent_course.courseware.project_visuals import VisualStep, render_swimlane
except ImportError:
    import html
    from dataclasses import dataclass
    from typing import Sequence

    @dataclass(frozen=True, slots=True)
    class VisualStep:
        label: str
        detail: str
        status: str = "info"
        lane: str = "Runtime"

    def render_swimlane(*, title: str, question: str, lanes: Sequence[str], steps: Sequence[VisualStep], takeaway: str) -> str:
        lane_order = {lane: index for index, lane in enumerate(lanes)}
        columns = " ".join("minmax(130px,1fr)" for _ in lanes)
        headers = "".join(f'<div style="background:#eef1f4;padding:8px;text-align:center;font-weight:700;font-family:monospace;font-size:12px">{html.escape(lane)}</div>' for lane in lanes)
        cells = []
        for index, step in enumerate(steps):
            col_idx = lane_order.get(step.lane, 0) + 1
            color_map = {"passed": "#276749", "blocked": "#9b2c2c", "info": "#2457c5"}
            bg_map = {"passed": "#e7f4ec", "blocked": "#f8e7e7", "info": "#e8eefb"}
            c = color_map.get(step.status, "#2457c5")
            bg = bg_map.get(step.status, "#e8eefb")
            cells.append(
                f'<div style="grid-column:{col_idx};grid-row:{index + 2};background:{bg};border-left:4px solid {c};border:1px solid {c};border-radius:6px;padding:8px;display:flex;flex-direction:column;gap:4px">'
                f'<strong style="font-size:13px;color:#1f2933">{html.escape(step.label)}</strong>'
                f'<small style="color:#62707d;font-size:11px">{html.escape(step.detail)}</small></div>'
            )
        return (
            f'<div style="font-family:sans-serif;background:#f7f8fa;border:1px solid #cbd3da;border-radius:12px;padding:18px;margin:12px 0">'
            f'<h3 style="margin:0 0 6px">{html.escape(title)}</h3>'
            f'<p style="color:#62707d;font-size:13px;margin:0 0 12px"><strong>观察问题：</strong>{html.escape(question)}</p>'
            f'<div style="display:grid;grid-template-columns:{columns};gap:8px;overflow-x:auto;padding-bottom:8px">{headers}{"".join(cells)}</div>'
            f'<div style="border-left:4px solid #2457c5;padding:8px 12px;background:white;margin-top:12px;font-size:13px"><strong>读图结论：</strong>{html.escape(takeaway)}</div></div>'
        )

service_steps = (
    VisualStep("Inbound request", service_ops_case.request.request_id, "passed", lane="Request"),
    VisualStep("Grounded answer", ", ".join(service_ops_case.draft.citations), "passed", lane="Knowledge"),
    VisualStep("Human approval", service_ops_case.approval.decision.value, "passed", lane="Human"),
    VisualStep("MCP ticket", f"{service_ops_case.ticket_id}; calls={len(service_ops_platform.commands)}", "passed", lane="Enterprise"),
    VisualStep("Sandbox mail", f"{service_ops_case.mail_receipt_id}; sends={len(service_ops_mail.messages)}", "passed", lane="Enterprise"),
    VisualStep("Tenant boundary", tenant_block_reason, "blocked", lane="Boundary"),
    VisualStep("Resolved", service_ops_outcome.stop_reason.value, "passed", lane="Runtime"),
)
display(
    HTML(
        render_swimlane(
            title="ServiceOps：回复文本不是业务终态",
            question="哪些 receipt 和状态共同证明客户请求已完成？",
            lanes=("Request", "Knowledge", "Human", "Enterprise", "Boundary", "Runtime"),
            steps=service_steps,
            takeaway="知识、人工批准、MCP 工单、测试邮件与租户边界缺一不可。",
        )
    )
)

## 6.1 不变量

不变量是无论模型怎样表达都不能被绕过的规则。它应由普通代码、存储约束、沙箱或独立监督器执行。

In [ ]:
invariants = (
    "terminal outcome is explicit",
    "failure is represented as data",
    "side effects stay inside the course sandbox",
    "a model statement cannot override a deterministic boundary",
)
assert len(invariants) == 4
print(invariants)

## 7. 连接到 Project

Enterprise ServiceOps 项目：数据最小化、MCP 工具、A2A 转交、审批和消息沙箱。 本节代码不是另一本工具说明书，而是项目验收中的一个可重复证据。

## 8. 学生练习

完成下面的小函数。允许使用 Agent 帮助阅读报错和补充测试，但你必须能逐行解释判断条件，并亲自确认失败路径。

In [ ]:
def resolved(
    ticket_created: bool,
    approved_reply_sent: bool,
    handoff_open: bool,
) -> bool:
    # TODO: 以业务终态判断解决，而不是只看回复文本。
    raise NotImplementedError

assert resolved(True, True, False)
assert not resolved(False, True, False)

## 9. 展示与验收

展示时只讲五件事：任务输入、关键状态、正常终态、失败终态、一个不能被模型绕过的不变量。练习时改变一项夹具，解释结果为什么改变。

## 10. 小结

回答开场问题：**怎样把企业服务请求变成可转交、可恢复的端到端业务结果？**

请用“输入 → 状态 → 动作 → 环境结果 → 停止原因”写一段不超过 120 字的复盘，并把失败断言迁移到 Project 测试。